# **Proyecto NLP**

* Aplicar un flujo básico de procesamiento de lenguaje natural (NLP) para resolver un problema de clasificación.
* Objetivo: Queremos implementar un sistema que sea capaz de detectar automáticamente si una página web contiene spam o no basándonos en su URL.

## **Imports**

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
import joblib

In [30]:
df = pd.read_csv("https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv")
df.head()


,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True


> **Observaciones:**  
> * Columnas:  
> - url = texto  
> - is_spam = etiqueta (¿el url pertenece a un spam? ¿si o no?)

## **Procesamiento de datos**

### **Limpiar la data**

In [31]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-zA-Z]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df["url_limpio"] = df["url"].apply(limpiar_texto)

### **Declarar variables**

In [32]:
X_text = df["url_limpio"]
y = df["is_spam"]

* **Vectorizar el texto**

In [38]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(X_text)
print(vectorizer.get_feature_names_out())
print(X.toarray())

['aa' 'aab' 'aaron' ... 'zwift' 'zwn' 'zyguxzjc']
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [ ]:
vect_df = pd.DataFrame(X.todense(), columns=vectorizer.get_feature_names_out())
vect_df["is_spam"] = y # vect_df["Resultado Esperado"] = df["is_spam"]

vect_df.head()

,aa,aab,aaron,ab,abacus,abandoned,abba,abbott,abbreviated,abc,...,zskl,ztz,zuck,zuckerberg,zuihitsu,zulalimtm,zwift,zwn,zyguxzjc,is_spam
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,True
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,True
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,True
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,False
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,True


### **Split data train - test**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## **ML**

### **Naive Bayes**

* **Crear y entrenar del modelo**

In [ ]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)

* **Predicciones:**

In [ ]:
y_pred = model.predict(X_test)

* **Evaluación del modelo (Accuracy)**:

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

* **Test data vs Data predicha**

In [ ]:
resultados = pd.DataFrame({    "Real": y_test,"Prediccion": y_pred})
resultados.head()

### **Modelo SVM**

* **Crear y entrenar el modelo**

In [ ]:
svm_model = SVC()

In [ ]:
svm_model.fit(X_train, y_train)

* **Predicciones:**

In [ ]:
y_pred_svm = svm_model.predict(X_test)

* **Evaluación del modelo (Accuracy)**:

In [ ]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("Accuracy SVM:", accuracy_svm)

### **Optimización del modelo (GridSearch)**

In [ ]:
param_grid = {'C': [0.1, 1, 10],'kernel': ['linear', 'rbf'],'gamma': ['scale', 'auto']}
grid = GridSearchCV(SVC(),param_grid,cv=5,scoring='accuracy',n_jobs=-1)
grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor score:", grid.best_score_)

In [ ]:
best_svm = grid.best_estimator_
y_pred_best = best_svm.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)

print("Accuracy optimizado:", accuracy_best)

### **Guardado del modelo**

In [ ]:
joblib.dump(best_svm, "svm_spam_model.pkl")

In [ ]:
modelo_cargado = joblib.load("svm_spam_model.pkl")

## **Observaciones finales:**


* Preparación de los datos: limpieza del texto, transformando todas las URLs a minúsculas y eliminando símbolos para separar las palabras que componen cada dirección los modelos de machine learning no pueden trabajar directamente con texto sin procesar.
* División del dataset en conjuntos de entrenamiento y prueba para poder evaluar el rendimiento del modelo de forma objetiva. 
* Se entrena también un modelo de clasificación Naive Bayes, que es especialmente adecuado para problemas de texto debido a su simplicidad y buen rendimiento en tareas de clasificación. Y se evalua el modelo utilizando el conjunto de prueba y se obtuvo una métrica de precisión (accuracy) que permite estimar qué tan bien el modelo es capaz de distinguir entre URLs spam y no spam.